In [1]:
import setup
setup.init()

In [2]:
from mypermit import permit_client

permit =permit_client

In [3]:
await permit.api.users.list()

PaginatedResultUserRead(data=[UserRead(key='4', id=UUID('1dc88613-442b-4e72-a194-63c333b77530'), organization_id=UUID('097fb5ac-4d7a-408d-a5f7-209528c54585'), project_id=UUID('a2ae8c71-1df6-486b-8302-7c0d95dc297a'), environment_id=UUID('0fa9b7a0-8a01-434b-ae83-81a8074d54e1'), associated_tenants=[], roles=[], created_at=datetime.datetime(2026, 10, 23, 7, 58, 34, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2026, 10, 23, 8, 6, 22, tzinfo=datetime.timezone.utc), email=None, first_name='RamChandra', last_name=None, attributes=None), UserRead(key='1', id=UUID('2ffefdea-7682-457f-92c0-43b6528b1101'), organization_id=UUID('097fb5ac-4d7a-408d-a5f7-209528c54585'), project_id=UUID('a2ae8c71-1df6-486b-8302-7c0d95dc297a'), environment_id=UUID('0fa9b7a0-8a01-434b-ae83-81a8074d54e1'), associated_tenants=[UserInTenant(tenant='default', roles=['viewer'], status=<UserStatus.pending: 'pending'>, resource_instance_roles=[])], roles=[UserRole(role='viewer', tenant='default')], created_at=da

In [4]:
from django.contrib.auth import get_user_model
User = get_user_model()

In [5]:
for u in User.objects.all():
    # print(u.id, u.username)
    # tenant_name= 'cfe'
    user_data={
        "key": f"{u.id}",
        "first_name": f"{u.username}"
    }
    response = await permit.api.users.sync(user_data)
    print(response)

key='1' id=UUID('2ffefdea-7682-457f-92c0-43b6528b1101') organization_id=UUID('097fb5ac-4d7a-408d-a5f7-209528c54585') project_id=UUID('a2ae8c71-1df6-486b-8302-7c0d95dc297a') environment_id=UUID('0fa9b7a0-8a01-434b-ae83-81a8074d54e1') associated_tenants=[UserInTenant(tenant='default', roles=['viewer'], status=<UserStatus.pending: 'pending'>, resource_instance_roles=[])] roles=[UserRole(role='viewer', tenant='default')] created_at=datetime.datetime(2026, 10, 23, 7, 58, 31, tzinfo=datetime.timezone.utc) updated_at=datetime.datetime(2026, 10, 23, 8, 6, 18, tzinfo=datetime.timezone.utc) email=None first_name='sarojdevkota' last_name=None attributes=None
key='2' id=UUID('c98422bb-5655-45cf-8922-e0f9d8fc4f10') organization_id=UUID('097fb5ac-4d7a-408d-a5f7-209528c54585') project_id=UUID('a2ae8c71-1df6-486b-8302-7c0d95dc297a') environment_id=UUID('0fa9b7a0-8a01-434b-ae83-81a8074d54e1') associated_tenants=[] roles=[] created_at=datetime.datetime(2026, 10, 23, 7, 58, 32, tzinfo=datetime.timezone.u

In [6]:
await permit.check("1","read","document")

True

In [7]:
resources = [
    {
        "name": "Document",
        "key": "document",
        "actions": {
            # "search": {},
            # "list": {},
            # "get": {},
            "read": {},
            "create": {},
            "update": {},
            "delete": {},
        }
    },
    {
        "name": "Movie Discovery Resource",
        "key": "movie_discovery_agent",
        "actions": {
                    "search": {},
                    "detail": {},          
                }
    }
]

In [8]:
from permit import  PermitApiError

async def get_or_create_resource(resource_data:dict):
    resource_key = resource_data.get('key')
    try:
        _resource = await permit.api.resources.get(resource_key)
       
    except PermitApiError as e:
        if e.status_code == 404:     
            _resource = await permit.api.resources.create(resource_data)
        else:
            raise e
    return _resource

In [9]:
for resource_data in resources:
    response = await get_or_create_resource(resource_data)
    print(response)

key='document' id=UUID('ed8683ed-e52d-4c7c-9c41-b7e112d7b2ec') organization_id=UUID('097fb5ac-4d7a-408d-a5f7-209528c54585') project_id=UUID('a2ae8c71-1df6-486b-8302-7c0d95dc297a') environment_id=UUID('0fa9b7a0-8a01-434b-ae83-81a8074d54e1') created_at=datetime.datetime(2026, 10, 23, 8, 56, 59, tzinfo=datetime.timezone.utc) updated_at=datetime.datetime(2026, 10, 23, 8, 56, 59, tzinfo=datetime.timezone.utc) name='Document' urn='prn:saroj-521:default:document' description=None actions={'delete': ActionBlockRead(name='delete', description=None, attributes={}, v1compat_path=None, v1compat_name=None, id=UUID('97ad4e9d-935f-4fd2-903c-11ec58d13cba'), key='delete'), 'read': ActionBlockRead(name='read', description=None, attributes={}, v1compat_path=None, v1compat_name=None, id=UUID('509b42fd-6931-49d3-a6a8-12c531148266'), key='read'), 'update': ActionBlockRead(name='update', description=None, attributes={}, v1compat_path=None, v1compat_name=None, id=UUID('b907c906-dbc5-4a98-91dc-8a6da6119323'), 

In [10]:
#  readonly_permissions=[
#     "document:read",
#     "movie_discovery:search",
    
#     "movie_discovery:detail",
# ]

In [11]:
roles =[
    {
        "name":"Manager",
        "key":"manager",
        "permissions":[
    "document:read",
    "document:create",
    "document:update",
    "document:delete",
    "movie_discovery_agent:search",
    "movie_discovery_agent:detail",
        ]
    
    },
    {
        "name":"Viewer",
        "key":"viewer",
        "permissions":[
            "document:read",
            "movie_discovery_agent:search",
            "movie_discovery_agent:detail",
                ]
    },
    
    
]

In [12]:
from permit import  PermitApiError

async def get_or_create_role(role_data:dict):
    _key = role_data.get('key')
    try:
        _resource = await permit.api.roles.get(_key)
       
    except PermitApiError as e:
        if e.status_code == 404:     
            _resource = await permit.api.roles.create(role_data)
        else:
            raise e
    return _resource

In [13]:
for role_data in roles:
    response = await get_or_create_role(role_data)
    print(response)


name='Manager' description=None permissions=['document:update', 'document:delete', 'document:read', 'movie_discovery_agent:detail', 'document:create', 'movie_discovery_agent:search'] attributes={} extends=[] granted_to=None v1compat_settings=None v1compat_attributes=None key='manager' id=UUID('cce25bfe-c5da-4700-8072-158cb7e5b8d7') organization_id=UUID('097fb5ac-4d7a-408d-a5f7-209528c54585') project_id=UUID('a2ae8c71-1df6-486b-8302-7c0d95dc297a') environment_id=UUID('0fa9b7a0-8a01-434b-ae83-81a8074d54e1') created_at=datetime.datetime(2026, 10, 23, 9, 23, tzinfo=datetime.timezone.utc) updated_at=datetime.datetime(2026, 10, 23, 9, 23, tzinfo=datetime.timezone.utc)
name='Viewer' description=None permissions=['document:read', 'movie_discovery_agent:search', 'movie_discovery_agent:detail'] attributes={} extends=[] granted_to=None v1compat_settings=None v1compat_attributes=None key='viewer' id=UUID('23bad866-6434-4124-aa37-1d57efe6ed27') organization_id=UUID('097fb5ac-4d7a-408d-a5f7-209528c5

In [14]:
user = User.objects.all().first()

In [15]:
response = await permit.api.users.assign_role({
    "user":f"{user.id}",
    "role":"viewer",
    "tenant":"default"
    
})
response

RoleAssignmentRead(id=UUID('d8060030-6c79-4389-9e64-739a27d28156'), user='1', role='viewer', tenant='default', resource_instance=None, resource_instance_id=None, user_id=UUID('2ffefdea-7682-457f-92c0-43b6528b1101'), role_id=UUID('23bad866-6434-4124-aa37-1d57efe6ed27'), tenant_id=UUID('f6663945-915e-496f-a33f-7295dc3e7be1'), organization_id=UUID('097fb5ac-4d7a-408d-a5f7-209528c54585'), project_id=UUID('a2ae8c71-1df6-486b-8302-7c0d95dc297a'), environment_id=UUID('0fa9b7a0-8a01-434b-ae83-81a8074d54e1'), created_at=datetime.datetime(2026, 10, 23, 9, 31, 1, tzinfo=datetime.timezone.utc))

In [16]:
await permit.check(f"{user.id}", "read", "document")

True

In [17]:
from ai.supervisors import get_supervisor

In [18]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer= InMemorySaver()

In [ ]:
supervisor =get_supervisor(model= "gemini-3.5-flash", checkpointer=checkpointer)

In [20]:
doc_id=4
user_id=user.id

In [24]:
import uuid
config={"configurable":{"user_id":user_id, "thread_id":uuid.uuid4()}}
response= supervisor.invoke({"messages":[{"role":"user","content":"delete any random document"}]},config)

c:\Users\User\Desktop\AI\django-ai-agent\.env\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\User\Desktop\AI\django-ai-agent\.env\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [22]:

# for i,msg in enumerate(response['messages']):
#     if msg.content:
#         print(i, msg.content, msg)
#         print("/n/n")

In [23]:
response['messages'][-1].content

[{'type': 'text',
  'text': 'Here are your recent documents:\n\n1. **LOTR** (ID: 6)\n2. **mission first** (ID: 5)\n3. **My Project Notes** (ID: 3)\n4. **testtitle** (ID: 2)\n5. **jhgkjgjbljb** (ID: 1)\n\nLet me know if you would like to view, edit, or perform any other actions with these documents!',
  'extras': {'signature': 'EqABCp0BAWkUfRN7I1YJj1k/rR8UAcMXkCIb72kSz9Av0yp4e2+iZ89IpjJbQU21z05XGxpMdzzUCHgvz6l677bHiYrUw5qJpRrLPazDCRkYaZb008jJvGxyj2HU3n2gJr5nNp2e+4qnYLNouHD5dZhaXtkH62VV8M428lCUyrScMDdUj0QUHsfsXTfahq+/XErxLej0XIb0qulZHE2QuwJi+w=='}}]